# Explore AdventureWorksLT

Before an agent writes SQL against a database, you should be able to read that database yourself. This notebook connects to the deployed Azure SQL sample database, walks its schema, runs a handful of small read-only queries, and then shows how the same metadata becomes the context the Text-to-SQL agent works from.

Everything runs through the production package under `src/`. The notebook opens no connection of its own and carries no second copy of the SQL validator, so what you see here is what the agent sees.

**Prerequisites:** `azd up` has provisioned the environment, `.env` is present, `az login` has been run, and ODBC Driver 18 for SQL Server is installed. If the driver is missing, every database step reports itself as skipped and the notebook still runs to the last cell.

## 1. Load typed settings

Configuration is read once, through `load_settings()`. Nothing below reads an environment variable directly. The target is printed through `DatabaseTarget.display`, which is built to be safe to show: it carries a server name, a database name, and an authentication mode, and never a token or a password.

In [4]:
from enterprise_agents_on_foundry.agents.nodes import answer_question, production_dependencies
from enterprise_agents_on_foundry.agents.prompts import sql_user_prompt
from enterprise_agents_on_foundry.agents.schema_context import load_schema_context
from enterprise_agents_on_foundry.agents.state import AgentInput
from enterprise_agents_on_foundry.config.settings import load_settings
from enterprise_agents_on_foundry.database.connection import DatabaseClient, connect, installed_odbc_drivers
from enterprise_agents_on_foundry.database.metadata import list_schemas, list_tables
from enterprise_agents_on_foundry.database.models import REQUIRED_ODBC_DRIVER, QueryResult
from enterprise_agents_on_foundry.database.validation import assert_read_only_sql, resolve_database_target
from enterprise_agents_on_foundry.errors import EaofError

settings = load_settings()
target = resolve_database_target(settings)

print(f"target:        {target.display}")
print(f"row limit:     {settings.database_max_result_rows}")
print(f"query timeout: {settings.database_query_timeout_seconds}s")
print(f"driver 18:     {'present' if REQUIRED_ODBC_DRIVER in installed_odbc_drivers() else 'missing'}")

target:        sql-eaof-dev-wgi4fh.database.windows.net/AdventureWorksLT (auth=entra)
row limit:     500
query timeout: 30s
driver 18:     present


## 2. Connect and check health

`connect()` resolves the target, acquires a Microsoft Entra token, and hands that token to the driver as a connection attribute rather than embedding it in a connection string. There is no username and no password anywhere in this path.

`health_check()` returns its outcome instead of raising, because a health check that raises cannot be used to report health. The reported latency is one round trip to Azure SQL, so treat it as a floor for every query that follows.

In [5]:
import pyodbc

print(pyodbc.drivers())

['SQL Server', 'ODBC Driver 18 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']


In [6]:
client: DatabaseClient | None = None
connection_error = ""

try:
    client = connect(settings)
except EaofError as error:
    connection_error = str(error).splitlines()[0]

if client is None:
    print(f"database unavailable: {connection_error}")
else:
    print(client.health_check().summary)

Connected to sql-eaof-dev-wgi4fh.database.windows.net/AdventureWorksLT (auth=entra) in 2809.2 ms.


One small display helper keeps the rest of the notebook readable. It adds no query logic: it calls `client.execute_sql()`, which validates the statement and truncates the result, then prints the rows through `QueryResult.format_table()`.

In [4]:
def show(sql: str, *, label: str, limit: int = 5) -> QueryResult | None:
    """Run one read-only query through the production client and print a small table."""
    if client is None:
        print(f"skipped: {connection_error}")
        return None
    result = client.execute_sql(sql, max_rows=limit, label=label)
    print(result.format_table(limit=limit))
    print(f"({result.row_count} row(s) in {result.elapsed_ms} ms)")
    return result

## 3. List the schemas

A schema is a namespace for tables. AdventureWorksLT puts everything interesting in `SalesLT` and leaves `dbo` almost empty, which is why every query below is schema qualified. The agent is instructed to qualify names for the same reason.

In [5]:
if client is None:
    print(f"skipped: {connection_error}")
else:
    for schema in list_schemas(client):
        print(f"{schema.name:<12} {schema.table_count} table(s)")

SalesLT      10 table(s)
dbo          2 table(s)


## 4. List the tables

AdventureWorksLT is a trimmed version of the full AdventureWorks sample. It has roughly a dozen tables covering four areas.

| Area | Tables | What it holds |
| --- | --- | --- |
| Customers | `Customer`, `CustomerAddress` | Companies and contacts that place orders. |
| Addresses | `Address` | Street addresses linked to customers. |
| Products | `Product`, `ProductCategory`, `ProductModel`, `ProductDescription` | The catalogue, its category tree, and localized descriptions. |
| Sales | `SalesOrderHeader`, `SalesOrderDetail` | One row per order, and one row per line within an order. |

Row counts come from partition metadata rather than from `COUNT(*)`, so they are approximate and cheap.

In [6]:
if client is None:
    print(f"skipped: {connection_error}")
else:
    for table in list_tables(client):
        print(f"{table.qualified_name:<40} ~{table.approximate_rows:>6} rows")

SalesLT.Customer                         ~   847 rows
SalesLT.ProductDescription               ~   762 rows
SalesLT.ProductModelProductDescription   ~   762 rows
SalesLT.SalesOrderDetail                 ~   542 rows
SalesLT.Address                          ~   450 rows
SalesLT.CustomerAddress                  ~   417 rows
SalesLT.Product                          ~   295 rows
SalesLT.ProductModel                     ~   128 rows
SalesLT.ProductCategory                  ~    41 rows
SalesLT.SalesOrderHeader                 ~    32 rows
dbo.BuildVersion                         ~     1 rows
dbo.ErrorLog                             ~     0 rows


## 5. Inspect columns, types, and keys

`load_schema_context()` runs one query over the system catalogue and renders every column with its type, its nullability, whether it belongs to the primary key, and which column a foreign key points at.

Foreign keys are the part that matters most. A model asked to join `SalesOrderDetail` to `Product` from column names alone will sometimes invent a join condition. Given the arrow notation below, it does not have to guess.

In [7]:
schema_context = ""
wanted = ("SalesLT.ProductCategory", "SalesLT.SalesOrderDetail")

if client is None:
    print(f"skipped: {connection_error}")
else:
    schema_context = load_schema_context(client)
    for block in schema_context.split("\n\n"):
        if block.split("\n", 1)[0] in wanted:
            print(block)
            print()

SalesLT.ProductCategory
  ProductCategoryID int not null primary key
  ParentProductCategoryID int -> SalesLT.ProductCategory.ProductCategoryID
  Name Name not null
  rowguid uniqueidentifier not null
  ModifiedDate datetime not null

SalesLT.SalesOrderDetail
  SalesOrderID int not null primary key -> SalesLT.SalesOrderHeader.SalesOrderID
  SalesOrderDetailID int not null primary key
  OrderQty smallint not null
  ProductID int not null -> SalesLT.Product.ProductID
  UnitPrice money not null
  UnitPriceDiscount money not null
  LineTotal numeric not null
  rowguid uniqueidentifier not null
  ModifiedDate datetime not null



## 6. Preview a few rows

Reading three rows from a table teaches you more about its shape than reading its column list. Columns are named explicitly rather than selected with `*`, so the output stays narrow enough to read.

In [10]:
previews = {
    "customers": "SELECT TOP (3) CustomerID, CompanyName, City FROM SalesLT.Customer ORDER BY CustomerID;",
    "products": "SELECT TOP (3) ProductID, Name, ListPrice FROM SalesLT.Product ORDER BY ProductID;",
    "orders": ("SELECT TOP (3) SalesOrderID, OrderDate, TotalDue FROM SalesLT.SalesOrderHeader ORDER BY SalesOrderID;"),
}

for label, sql in previews.items():
    print(f"-- {label}")
    if label == "customers":
        sql = (
            "SELECT TOP (3) c.CustomerID, c.CompanyName, addr.City "
            "FROM SalesLT.Customer AS c "
            "OUTER APPLY ("
            "  SELECT TOP (1) a.City "
            "  FROM SalesLT.CustomerAddress AS ca "
            "  JOIN SalesLT.Address AS a ON a.AddressID = ca.AddressID "
            "  WHERE ca.CustomerID = c.CustomerID "
            "  ORDER BY ca.AddressID"
            ") AS addr "
            "ORDER BY c.CustomerID;"
        )
    show(sql, label=label, limit=3)
    print()

-- customers
CustomerID  CompanyName               City
----------  ------------------------  ----
1           A Bike Store              None
2           Progressive Sports        None
3           Advanced Bike Components  None
(3 row(s) in 93.8 ms)

-- products
ProductID  Name                       ListPrice
---------  -------------------------  ---------
680        HL Road Frame - Black, 58  1431.5000
706        HL Road Frame - Red, 58    1431.5000
707        Sport-100 Helmet, Red      34.9900  
(3 row(s) in 68.2 ms)

-- orders
SalesOrderID  OrderDate            TotalDue  
------------  -------------------  ----------
71774         2008-06-01 00:00:00  972.7850  
71776         2008-06-01 00:00:00  87.0851   
71780         2008-06-01 00:00:00  42452.6519
(3 row(s) in 77.1 ms)



## 7. Six beginner queries

These are the questions people actually ask of a sales database, written as plainly as the schema allows. Each one is a single `SELECT`, each uses `TOP (n)` rather than `LIMIT` because this is T-SQL, and each is the sort of statement the agent should produce for the matching question in section 9.

In [12]:
queries = {
    "how many customers are there": "SELECT COUNT(*) AS customer_count FROM SalesLT.Customer;",
    "the five most expensive products": (
        "SELECT TOP (5) Name, ListPrice FROM SalesLT.Product ORDER BY ListPrice DESC;"
    ),
    "products per category": (
        "SELECT TOP (5) c.Name AS category, COUNT(*) AS product_count "
        "FROM SalesLT.Product AS p "
        "JOIN SalesLT.ProductCategory AS c ON c.ProductCategoryID = p.ProductCategoryID "
        "GROUP BY c.Name ORDER BY product_count DESC;"
    ),
    "the most recent sales orders": (
        "SELECT TOP (5) SalesOrderID, OrderDate, TotalDue FROM SalesLT.SalesOrderHeader ORDER BY OrderDate DESC;"
    ),
    "sales totals by product category": (
        "SELECT TOP (5) c.Name AS category, SUM(d.LineTotal) AS sales_total "
        "FROM SalesLT.SalesOrderDetail AS d "
        "JOIN SalesLT.Product AS p ON p.ProductID = d.ProductID "
        "JOIN SalesLT.ProductCategory AS c ON c.ProductCategoryID = p.ProductCategoryID "
        "GROUP BY c.Name ORDER BY sales_total DESC;"
    ),
    "which customers placed the largest orders": (
        "SELECT TOP (5) cu.CompanyName, h.SalesOrderID, h.TotalDue "
        "FROM SalesLT.SalesOrderHeader AS h "
        "JOIN SalesLT.Customer AS cu ON cu.CustomerID = h.CustomerID "
        "ORDER BY h.TotalDue DESC;"
    ),
}

for label, sql in queries.items():
    print(f"-- {label}")
    show(sql, label=label, limit=5)
    print()

-- how many customers are there
customer_count
--------------
847           
(1 row(s) in 58.6 ms)

-- the five most expensive products
Name              ListPrice
----------------  ---------
Road-150 Red, 62  3578.2700
Road-150 Red, 44  3578.2700
Road-150 Red, 48  3578.2700
Road-150 Red, 52  3578.2700
Road-150 Red, 56  3578.2700
(5 row(s) in 58.5 ms)

-- products per category
category         product_count
---------------  -------------
Road Bikes       43           
Road Frames      33           
Mountain Bikes   32           
Mountain Frames  28           
Touring Bikes    22           
(5 row(s) in 63.1 ms)

-- the most recent sales orders
SalesOrderID  OrderDate            TotalDue   
------------  -------------------  -----------
71784         2008-06-01 00:00:00  119960.8240
71783         2008-06-01 00:00:00  92663.5609 
71782         2008-06-01 00:00:00  43962.7901 
71780         2008-06-01 00:00:00  42452.6519 
71776         2008-06-01 00:00:00  87.0851    
(5 row(s) in 61.2 m

## 8. How schema metadata becomes agent context

The legacy agent discovered the schema by calling `list_tables` and `get_schema` tools, which cost at least two model turns per question and let the model choose which parts of the database to look at.

AdventureWorksLT is small enough that the whole schema fits in one prompt. The rendering you saw in section 5 is loaded once per run and pasted into the prompt, so the model never chooses what it can see and never pays for a discovery turn. The cell below shows how large that context actually is, and where it lands inside the prompt.

In [13]:
if not schema_context:
    print(f"skipped: {connection_error}")
else:
    lines = schema_context.splitlines()
    tables = [line for line in lines if line and not line.startswith(" ")]
    print(f"{len(tables)} tables, {len(lines)} lines, {len(schema_context)} characters")
    print()
    prompt = sql_user_prompt(
        question="Which product categories sell the most?",
        schema_context=schema_context,
        max_rows=5,
    )
    print(prompt[:600])
    print("...")

12 tables, 132 lines, 4001 characters

Schema:
dbo.BuildVersion
  SystemInformationID tinyint not null primary key
  Database Version nvarchar not null
  VersionDate datetime not null
  ModifiedDate datetime not null

dbo.ErrorLog
  ErrorLogID int not null primary key
  ErrorTime datetime not null
  UserName sysname not null
  ErrorNumber int not null
  ErrorSeverity int
  ErrorState int
  ErrorProcedure nvarchar
  ErrorLine int
  ErrorMessage nvarchar not null

SalesLT.Address
  AddressID int not null primary key
  AddressLine1 nvarchar not null
  AddressLine2 nvarchar
  City nvarchar not null
  StateProvince Name not null
  Count
...


## 9. One question, traced end to end

A question travels through seven stages, and only two of them are model calls. The rest is deterministic Python, which is the whole point of the v0.3 graph.

The cell below walks the stages with a fixed example statement, so you can read the validation and execution steps without waiting for a model. The cell after it runs the real agent on the same question.

In [14]:
question = "Which five product categories have the highest sales?"
candidate_sql = (
    "SELECT TOP (5) c.Name AS category, SUM(d.LineTotal) AS sales_total "
    "FROM SalesLT.SalesOrderDetail AS d "
    "JOIN SalesLT.Product AS p ON p.ProductID = d.ProductID "
    "JOIN SalesLT.ProductCategory AS c ON c.ProductCategoryID = p.ProductCategoryID "
    "GROUP BY c.Name ORDER BY sales_total DESC;"
)

print(f"1. question:  {question}")
print("2. tables:    SalesLT.SalesOrderDetail -> SalesLT.Product -> SalesLT.ProductCategory")
print(f"3. sql:       {candidate_sql}")
print(f"4. validated: {len(assert_read_only_sql(candidate_sql))} characters, one read-only statement")
print("5. execution:")
trace_result = show(candidate_sql, label="trace", limit=5)
print(f"6. rows:      {trace_result.row_count if trace_result else 0}")
print("7. answer:    written by the model from those rows, in the compose_answer node")

1. question:  Which five product categories have the highest sales?
2. tables:    SalesLT.SalesOrderDetail -> SalesLT.Product -> SalesLT.ProductCategory
3. sql:       SELECT TOP (5) c.Name AS category, SUM(d.LineTotal) AS sales_total FROM SalesLT.SalesOrderDetail AS d JOIN SalesLT.Product AS p ON p.ProductID = d.ProductID JOIN SalesLT.ProductCategory AS c ON c.ProductCategoryID = p.ProductCategoryID GROUP BY c.Name ORDER BY sales_total DESC;
4. validated: 277 characters, one read-only statement
5. execution:
category         sales_total  
---------------  -------------
Touring Bikes    220655.375796
Road Bikes       183130.296808
Mountain Bikes   170825.886000
Mountain Frames  54949.602000 
Road Frames      24346.584000 
(5 row(s) in 60.0 ms)
6. rows:      5
7. answer:    written by the model from those rows, in the compose_answer node


Now the same question through the real graph. `answer_question()` loads the schema, drafts SQL, validates it, executes it, and writes the answer, with at most one repair attempt and never more than two statements in total.

In [15]:
if client is None:
    print(f"skipped: {connection_error}")
else:
    try:
        output = answer_question(
            production_dependencies(settings, client),
            AgentInput(question=question, max_rows=5),
        )
    except EaofError as error:
        print(f"agent unavailable: {error}")
    else:
        print(f"status:  {output.status}")
        print(f"repairs: {output.repair_attempts}")
        print(f"sql:     {output.sql}")
        print(f"rows:    {output.row_count}")
        print(f"answer:  {output.answer}")

status:  succeeded
repairs: 0
sql:     SELECT TOP (5) pc.Name AS ProductCategory, SUM(sod.LineTotal) AS TotalSales FROM SalesLT.SalesOrderDetail AS sod INNER JOIN SalesLT.Product AS p ON sod.ProductID = p.ProductID INNER JOIN SalesLT.ProductCategory AS pc ON p.ProductCategoryID = pc.ProductCategoryID GROUP BY pc.Name ORDER BY SUM(sod.LineTotal) DESC
rows:    5
answer:  The five product categories with the highest sales are Touring Bikes (220,655.38), Road Bikes (183,130.30), Mountain Bikes (170,825.89), Mountain Frames (54,949.60), and Road Frames (24,346.58).


## 10. An unsafe write is rejected before it reaches Azure SQL

The read-only guarantee does not rest on the model behaving, or on prompt wording. Every statement passes `assert_read_only_sql()` first, and the database client calls it a second time immediately before handing anything to the driver.

Database permissions remain the real backstop. The agent identity holds `db_datareader` and nothing more, so even a validator bug fails closed.

In [12]:
unsafe = (
    "DELETE FROM SalesLT.Customer;",
    "UPDATE SalesLT.Product SET ListPrice = 0;",
    "SELECT 1; DROP TABLE SalesLT.Customer;",
    "EXEC sp_executesql N'SELECT 1';",
)

for statement in unsafe:
    try:
        assert_read_only_sql(statement)
    except EaofError as error:
        print(f"rejected: {statement}")
        print(f"          {error}")
    else:
        print(f"ACCEPTED, which is a bug: {statement}")
    print()

print("No connection was opened for any of the statements above.")

rejected: DELETE FROM SalesLT.Customer;
          Query must begin with SELECT or WITH; found 'DELETE'.

rejected: UPDATE SalesLT.Product SET ListPrice = 0;
          Query must begin with SELECT or WITH; found 'UPDATE'.

rejected: SELECT 1; DROP TABLE SalesLT.Customer;
          Query contains 2 statements; only a single statement is allowed.

rejected: EXEC sp_executesql N'SELECT 1';
          Query must begin with SELECT or WITH; found 'EXEC'.

No connection was opened for any of the statements above.


## 11. What the agent needs, and what this database cannot show

Five tables carry almost every question worth asking of AdventureWorksLT.

| Table | Why it matters |
| --- | --- |
| `SalesLT.Customer` | The entity most questions filter or group by. |
| `SalesLT.Product` | Names, prices, and the link into the category tree. |
| `SalesLT.ProductCategory` | The grouping label people use in questions, which never appears in `Product` itself. |
| `SalesLT.SalesOrderHeader` | Order dates and totals, so anything time based starts here. |
| `SalesLT.SalesOrderDetail` | Line level amounts, and the only place a per product total can be computed. |

Be honest about the limits before drawing conclusions from any of it.

* It holds a few thousand rows, so query latency here says nothing about how the agent behaves at production scale.
* Order dates are historical and tightly clustered, so questions about recent activity return old rows and time filters look artificially narrow.
* There is one currency, one region, and no cancelled or partial order flow, so the agent is never tested against the ambiguity a real sales schema carries.
* Column names are unusually clean, which flatters the model. Real schemas carry abbreviations and legacy names that make the schema context far more valuable than it appears here.
* Nothing here exercises row level security, masking, or tenant isolation, all of which change how a Text-to-SQL agent must be built.

The next notebook, `02_modern_langgraph_text_to_sql.ipynb`, builds the agent that consumes everything shown here.

In [13]:
if client is not None:
    client.close()
    print("connection closed")
else:
    print("nothing to close")

nothing to close
